# Exploratory analysis (reads saved outputs only)

This notebook does **not** train anything and no reported number depends on it. It reads the CSV/JSON files written by `scripts/` so the results can be browsed interactively. Run the scripts first (see README).

In [1]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
T, M = ROOT / 'results/tables', ROOT / 'results/metrics'
pd.set_option('display.width', 160)

## Core subgraph audit

In [2]:
audit = json.loads((M / 'dataset_audit.json').read_text())['core']
{k: audit[k] for k in audit if k.startswith('edge_homophily')}

{'edge_homophily_gender': 0.4492618202205195,
 'edge_homophily_gender_if_random_mixing': 0.5004205,
 'edge_homophily_region': 0.9633713324612222,
 'edge_homophily_region_if_random_mixing': 0.51445,
 'edge_homophily_label': 0.5858717996636142,
 'edge_homophily_label_if_random_mixing': 0.500648}

In [3]:
pd.read_csv(T / 'core_groups_gender_x_region.csv')

,group,n,positives,negatives,positive_rate,mean_degree
0,g0_r0,550,229,321,0.416364,4.472727
1,g0_r1,421,161,260,0.382423,6.567696
2,g1_r0,620,360,260,0.580645,4.396774
3,g1_r1,409,214,195,0.523227,6.726161


## Embedding diagnostics: can the protected attribute be read from the embedding?

In [4]:
pd.read_csv(T / 'table_embedding_diagnostics.csv')[['embedding','attribute','auroc_mean','auroc_std']]

,embedding,attribute,auroc_mean,auroc_std
0,fw,gender,0.501,0.014
1,fw,region,0.992,0.006
2,fwr,gender,0.511,0.013
3,fwr,region,0.986,0.009
4,n2v,gender,0.511,0.019
5,n2v,region,0.991,0.003


## Main comparison (default operating point; 5 seeds; block-bootstrap 95% CI)

In [5]:
pd.read_csv(T / 'table_methods_default.csv')[['method','macro_f1','dEOpp_g','dDP_g','brier']]

,method,macro_f1,dEOpp_g,dDP_g,brier
0,Feature LR,"0.681 [0.633, 0.709]","0.065 [0.047, 0.143]","0.162 [0.119, 0.202]","0.204 [0.187, 0.219]"
1,SGC (simple GNN),"0.685 [0.616, 0.725]","0.065 [0.033, 0.131]","0.061 [0.025, 0.122]","0.195 [0.175, 0.212]"
2,Node2Vec+LR,"0.667 [0.586, 0.709]","0.101 [0.057, 0.157]","0.034 [0.025, 0.073]","0.214 [0.195, 0.230]"
3,FairWalk+LR,"0.657 [0.576, 0.698]","0.087 [0.057, 0.157]","0.032 [0.021, 0.076]","0.215 [0.194, 0.233]"
4,Node2Vec+group thr.,"0.667 [0.598, 0.700]","0.013 [0.020, 0.148]","0.098 [0.049, 0.158]","0.214 [0.195, 0.230]"
5,"Node2Vec CMN, no pairwise","0.667 [0.586, 0.709]","0.101 [0.057, 0.157]","0.034 [0.025, 0.073]","0.214 [0.195, 0.230]"
6,"FairWalk CMN, no pairwise","0.657 [0.576, 0.698]","0.087 [0.057, 0.157]","0.032 [0.021, 0.076]","0.215 [0.194, 0.233]"
7,Node2Vec+uniform CMN,"0.678 [0.604, 0.717]","0.103 [0.063, 0.173]","0.025 [0.023, 0.073]","0.210 [0.189, 0.228]"
8,FairWalk+uniform CMN,"0.669 [0.592, 0.712]","0.089 [0.045, 0.157]","0.032 [0.020, 0.073]","0.211 [0.188, 0.232]"
9,Node2Vec+global scalar,"0.674 [0.597, 0.715]","0.105 [0.069, 0.168]","0.034 [0.024, 0.076]","0.210 [0.190, 0.228]"


## Confirmatory tests H1–H4 (Holm-adjusted)

In [6]:
pd.read_csv(T / 'confirmatory_tests.csv')[['id','a','b','estimate_mean_over_seeds','ci95_low','ci95_high','p_unadjusted','p_holm']]

,id,a,b,estimate_mean_over_seeds,ci95_low,ci95_high,p_unadjusted,p_holm
0,H1,fw_uniform,fw_lr,0.001629,-0.037986,0.021589,0.593407,1.0
1,H2,n2v_uniform,n2v_lr,0.001327,-0.028067,0.039372,0.787213,1.0
2,H3,fw_hca,fw_uniform,0.003677,-0.020156,0.025943,0.753247,1.0
3,H4,fw_hca,fw_global,0.000072,-0.036614,0.023221,0.811189,1.0


## Selected attenuation parameters per seed (stability check)

In [7]:
pd.read_csv(T / 'table_selected_parameters.csv')

,seed,emb,eta,global,lambda
0,11,n2v,0.0,1.0,0.25
1,11,fw,0.8,0.0,0.25
2,22,n2v,0.4,1.0,0.25
3,22,fw,0.0,1.0,0.25
4,33,n2v,0.0,1.0,0.25
5,33,fw,0.3,0.0,0.25
6,44,n2v,0.3,0.0,0.25
7,44,fw,0.5,0.0,0.25
8,55,n2v,0.2,0.0,0.25
9,55,fw,1.0,0.9,0.25


## Synthetic study and robustness (present only after `run_synthetic.py` / `run_robustness.py`)

In [8]:
for f in ['table_synthetic_homophily_degree.csv', 'table_robustness_paired_effects.csv']:
    p = T / f
    print(f, 'exists' if p.exists() else 'MISSING')
    if p.exists():
        display(pd.read_csv(p).head(12))

table_synthetic_homophily_degree.csv exists


,group_homophily,degree_ratio,n_real,real_hA,unary_dEOpp,uniform_dEOpp,hca_dEOpp,global_dEOpp,smooth_dEOpp_mean,smooth_dEOpp_sd,smooth_dDP_mean,smooth_dDP_sd,smooth_f1_mean,hca_vs_uniform_dEOpp,hca_vs_global_dEOpp,hca_vs_global_f1
0,0.5,1.0,10,0.4962,0.0261,0.0241,0.0323,0.0239,-0.0020,0.0214,-0.0085,0.0262,0.0117,0.0083,0.0085,-0.0076
1,0.5,2.0,10,0.4963,0.0261,0.0234,0.0445,0.0266,-0.0026,0.0174,-0.0127,0.0240,0.0132,0.0210,0.0179,-0.0037
2,0.5,4.0,10,0.4960,0.0261,0.0263,0.0429,0.0344,0.0002,0.0253,-0.0142,0.0310,0.0090,0.0166,0.0085,-0.0134
3,0.6,1.0,10,0.5960,0.0261,0.0505,0.0336,0.0394,0.0244,0.0438,0.0263,0.0409,0.0099,-0.0169,-0.0057,-0.0041
4,0.6,2.0,10,0.5962,0.0261,0.0308,0.0351,0.0220,0.0047,0.0214,0.0092,0.0186,0.0125,0.0043,0.0131,-0.0055
5,0.6,4.0,10,0.5956,0.0261,0.0211,0.0306,0.0238,-0.0050,0.0265,-0.0026,0.0353,0.0076,0.0096,0.0068,-0.0148
6,0.7,1.0,10,0.6982,0.0261,0.0578,0.0329,0.0379,0.0317,0.0443,0.0371,0.0390,0.0122,-0.0249,-0.0049,-0.0044
7,0.7,2.0,10,0.6981,0.0261,0.0574,0.0513,0.0310,0.0313,0.0373,0.0343,0.0286,0.0119,-0.0061,0.0203,-0.0073
8,0.7,4.0,10,0.6976,0.0261,0.0274,0.0303,0.0256,0.0013,0.0267,0.0100,0.0287,0.0098,0.0029,0.0047,-0.0040
9,0.8,1.0,10,0.7993,0.0261,0.0699,0.0438,0.0324,0.0438,0.0489,0.0534,0.0345,0.0122,-0.0261,0.0115,-0.0086


table_robustness_paired_effects.csv exists


,condition,embedding,metric,uniform_minus_lr_mean,uniform_minus_lr_sd,hca_minus_global_mean,hca_minus_global_sd,n_seeds
0,baseline,fw,brier,-0.0033,0.0020,-0.0017,0.0033,5
1,baseline,fw,dDP_gender,0.0000,0.0115,0.0038,0.0173,5
2,baseline,fw,dEOpp_gender,0.0016,0.0373,0.0001,0.0193,5
3,baseline,fw,macro_f1,0.0118,0.0049,-0.0003,0.0089,5
4,baseline,n2v,brier,-0.0040,0.0029,-0.0009,0.0013,5
5,baseline,n2v,dDP_gender,-0.0093,0.0236,-0.0064,0.0087,5
6,baseline,n2v,dEOpp_gender,0.0013,0.0326,0.0013,0.0045,5
7,baseline,n2v,macro_f1,0.0114,0.0189,0.0035,0.0128,5
8,corrupt0.1,fw,brier,-0.0023,0.0025,-0.0021,0.0022,5
9,corrupt0.1,fw,dDP_gender,0.0031,0.0249,-0.0053,0.0170,5
